In [1]:
import pandas as pd
import torch

# Per riproducibilità
torch.manual_seed(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("../data/all_ships.csv")


In [2]:
from sklearn.model_selection import train_test_split

X = df[["sex", "age", "age_missing", "class", "crew"]]
Y = df["survived"]

X_tensor = torch.tensor(X.values, dtype=torch.float32)
Y_tensor = torch.tensor(Y.values, dtype=torch.long)

# per riproducibilità si usa random_state fissato
X_train, X_test, Y_train, Y_test = train_test_split(X_tensor, Y_tensor, test_size=0.2, random_state=42, stratify=Y_tensor) 
Y_train = torch.where(Y_train == 1, 1, -1)
Y_test = torch.where(Y_test == 1, 1, -1)

mean = X_train.mean(0)
std = X_train.std(0)

X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std


In [3]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_norm, Y_train)
test_ds = TensorDataset(X_test_norm, Y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [4]:
from torch import nn

class LinearSVM(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.linear(x)


In [5]:
from torch.optim import SGD
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score

def hinge_loss(outputs, labels):
    return torch.mean(torch.clamp(1 - labels * outputs, min=0))

def train_model(model, train_loader, test_loader, lr=0.05, epochs=300):
    writer = SummaryWriter(f'../results/{model._get_name()}')
    model = model.to(device)

    optimizer = SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.001)

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        y_true = []
        y_pred = []
        
        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            output = model(X_batch)
            loss = hinge_loss(output.view(-1), Y_batch) # output e Y_batch devono avere stesso shape (batch_size,)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item() * X_batch.size(0)
            
            with torch.no_grad():
                preds = torch.sign(output.view(-1))
                preds[preds == 0] = 1
                y_true.extend(Y_batch.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(y_true, y_pred)

        model.eval()

        test_loss = 0.0
        y_true = []
        y_pred = []

        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)

                output = model(X_batch)
                loss = hinge_loss(output.view(-1), Y_batch) # output e Y_batch devono avere stesso shape (batch_size,)
                test_loss += loss.item() * X_batch.size(0)

                preds = torch.sign(output.view(-1))
                preds[preds == 0] = 1
                y_true.extend(Y_batch.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        test_loss /= len(test_loader.dataset)
        test_acc = accuracy_score(y_true, y_pred)

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('accuracy/train', train_acc, epoch)
        writer.add_scalar('loss/test', test_loss, epoch)
        writer.add_scalar('accuracy/test', test_acc, epoch)

        if epoch % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")

    print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")
    writer.close()

    return model, train_loss, train_acc, test_loss, test_acc

In [6]:
model = LinearSVM(in_features=5)
model, train_loss, train_acc, test_loss, test_acc = train_model(model, train_loader, test_loader)

Epoch 1/300 | train_loss 0.7371 | train_acc 0.6530 | test_loss 0.6934 | test_acc 0.6749
Epoch 51/300 | train_loss 0.6771 | train_acc 0.6755 | test_loss 0.7055 | test_acc 0.6749
Epoch 101/300 | train_loss 0.6839 | train_acc 0.6755 | test_loss 0.6724 | test_acc 0.6749
Epoch 151/300 | train_loss 0.6817 | train_acc 0.6755 | test_loss 0.6765 | test_acc 0.6749
Epoch 201/300 | train_loss 0.6778 | train_acc 0.6755 | test_loss 0.6861 | test_acc 0.6749
Epoch 251/300 | train_loss 0.6769 | train_acc 0.6755 | test_loss 0.6828 | test_acc 0.6749
Epoch 300/300 | train_loss 0.6745 | train_acc 0.6755 | test_loss 0.6764 | test_acc 0.6749


In [7]:
from sklearn.svm import SVC

svm_sklearn = SVC(kernel="linear", C=0.1, random_state=42)
svm_sklearn.fit(X_train, Y_train)

acc_sklearn = svm_sklearn.score(X_test, Y_test)
print(f"test_acc sklearn: {acc_sklearn:.4f}")


test_acc sklearn: 0.6749


In [8]:
print("Model Comparison\n")

print("LinearSVM")
print(f"train_loss: {train_loss:.4f}")
print(f"train_acc: {train_acc:.4f}")
print(f"test_loss: {test_loss:.4f}")
print(f"test_acc: {test_acc:.4f}\n")

print("SVM sklearn")
print(f"test_acc: {acc_sklearn:.4f}")


Model Comparison

LinearSVM
train_loss: 0.6745
train_acc: 0.6755
test_loss: 0.6764
test_acc: 0.6749

SVM sklearn
test_acc: 0.6749
